# Carga del conjunto de datos

In [18]:
import pandas as pd
import numpy as np
import unicodedata

In [19]:
# Función para limpiar y estandarizar nombres de columnas
def clean_column_name(col_name):
    # 1. Eliminar espacios al inicio/final, convertir a minúsculas
    cleaned_name = col_name.strip().lower()

    # 2. Eliminar tildes y otros caracteres diacríticos (ej. µ, °)
    # Normaliza la cadena y luego codifica/decodifica para eliminar los acentos
    cleaned_name = unicodedata.normalize('NFKD', cleaned_name).encode('ascii', 'ignore').decode('utf-8')

    # 3. Reemplazar símbolos específicos y unidades para mayor claridad
    cleaned_name = cleaned_name.replace('(', '').replace(')', '') # Eliminar paréntesis
    cleaned_name = cleaned_name.replace('°', '') # Eliminar símbolo de grado
    cleaned_name = cleaned_name.replace('ug/m3', 'ug_m3') # µg/m³ -> ug_m3 (después de eliminar µ)
    cleaned_name = cleaned_name.replace('%', 'percent') # % -> percent
    cleaned_name = cleaned_name.replace('w/m2', 'w_m2') # W/m² -> w_m2
    cleaned_name = cleaned_name.replace('m/s', 'm_s') # m/s -> m_s
    cleaned_name = cleaned_name.replace('mmhg', 'mm_hg') # mmHg -> mm_hg

    # 4. Reemplazar espacios, puntos y barras con guiones bajos
    cleaned_name = cleaned_name.replace(' ', '_')
    cleaned_name = cleaned_name.replace('.', '_') # Para 'PM2.5' -> 'pm2_5'
    cleaned_name = cleaned_name.replace('/', '_')

    # 5. Limpiar guiones bajos duplicados y los de inicio/final
    cleaned_name = cleaned_name.replace('__', '_').replace('__', '_').strip('_') # Ejecutar dos veces para seguridad

    return cleaned_name

In [21]:
file_path = 'Datasets/SISTEMA_DE_VIGILANCIA_DE_CALIDAD_DE_AIRE.csv'
df = pd.read_csv(file_path, sep=',', na_values=['NA', '-999']) 
print(f"Dataset '{file_path}' cargado exitosamente.")

Dataset 'Datasets/SISTEMA_DE_VIGILANCIA_DE_CALIDAD_DE_AIRE.csv' cargado exitosamente.


In [22]:
# Aplicar la función de limpieza a todos los nombres de columnas
df.columns = [clean_column_name(col) for col in df.columns]

In [23]:
print("\n--- Nombres de columnas estandarizados ---")
print(df.columns.tolist())


--- Nombres de columnas estandarizados ---
['fecha_y_hora', 'presion_atmosferica_mm_hg', 'temperatura_ambiente_celsius', 'velocidad_del_viento_m_s', 'direccion_del_viento', 'humedad_relativa_percent', 'radiacion_solar_w_m2', 'precipitacion_mm', 'no2_g_m3', 'no_g_m3', 'nox_g_m3', 'co_g_m3', 'o3_g_m3', 'so2_g_m3', 'trs_g_m3', 'pm10_g_m3', 'pm2_5_g_m3']


In [24]:
print("\n--- Primeras 5 filas del dataset ---")
print(df.head())


--- Primeras 5 filas del dataset ---
             fecha_y_hora  presion_atmosferica_mm_hg  \
0  01/01/2024 12:00:00 AM                    696.291   
1  01/01/2024 01:00:00 AM                    696.132   
2  01/01/2024 02:00:00 AM                    695.799   
3  01/01/2024 03:00:00 AM                    695.300   
4  01/01/2024 04:00:00 AM                    694.951   

   temperatura_ambiente_celsius  velocidad_del_viento_m_s  \
0                        24.086                     1.142   
1                        23.734                     1.420   
2                        23.169                     1.592   
3                        23.281                     1.368   
4                        22.887                     2.341   

   direccion_del_viento  humedad_relativa_percent  radiacion_solar_w_m2  \
0               181.289                    83.830                   0.0   
1               121.161                    82.638                   0.0   
2                68.523          

In [25]:
print("\n--- Información general del dataset ---")
df.info()


--- Información general del dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   fecha_y_hora                  8784 non-null   object 
 1   presion_atmosferica_mm_hg     8700 non-null   float64
 2   temperatura_ambiente_celsius  8703 non-null   float64
 3   velocidad_del_viento_m_s      8703 non-null   float64
 4   direccion_del_viento          8703 non-null   float64
 5   humedad_relativa_percent      8703 non-null   float64
 6   radiacion_solar_w_m2          8703 non-null   float64
 7   precipitacion_mm              7433 non-null   float64
 8   no2_g_m3                      0 non-null      float64
 9   no_g_m3                       0 non-null      float64
 10  nox_g_m3                      0 non-null      float64
 11  co_g_m3                       0 non-null      float64
 12  o3_g_m3              

In [26]:
print("\n--- Estadísticas descriptivas de columnas numéricas ---")
print(df.describe())


--- Estadísticas descriptivas de columnas numéricas ---
       presion_atmosferica_mm_hg  temperatura_ambiente_celsius  \
count                8700.000000                   8703.000000   
mean                  694.593785                     25.135718   
std                     1.272896                      2.798841   
min                   690.607000                     19.410000   
25%                   693.724000                     22.804000   
50%                   694.674000                     24.793000   
75%                   695.507250                     27.371500   
max                   698.510000                     32.580000   

       velocidad_del_viento_m_s  direccion_del_viento  \
count               8703.000000           8703.000000   
mean                   2.042907            197.559655   
std                    1.316536            124.636971   
min                    0.432000              0.045000   
25%                    1.047500             76.015500   
50%   

# Métricas Iniciales del Dataset  (Estado Sucio)

# 1. Número total de filas y columna

In [27]:
print(f"\nDimensiones del dataset: {df.shape[0]} filas, {df.shape[1]} columnas")


Dimensiones del dataset: 8784 filas, 17 columnas


# 2. Conteo de valores faltantes por columna

In [28]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing %': missing_percentage})
print("\nValores Faltantes por Columna:")
# Mostrar solo las columnas con valores faltantes
print(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing %', ascending=False))
if missing_df[missing_df['Missing Count'] > 0].empty:
    print("¡No hay valores faltantes en el dataset inicial!")


Valores Faltantes por Columna:
                              Missing Count   Missing %
no2_g_m3                               8784  100.000000
co_g_m3                                8784  100.000000
nox_g_m3                               8784  100.000000
no_g_m3                                8784  100.000000
so2_g_m3                               8784  100.000000
precipitacion_mm                       1351   15.380237
trs_g_m3                               1161   13.217213
pm10_g_m3                               378    4.303279
pm2_5_g_m3                              378    4.303279
o3_g_m3                                 258    2.937158
presion_atmosferica_mm_hg                84    0.956284
humedad_relativa_percent                 81    0.922131
temperatura_ambiente_celsius             81    0.922131
radiacion_solar_w_m2                     81    0.922131
velocidad_del_viento_m_s                 81    0.922131
direccion_del_viento                     81    0.922131


# 3. Número de filas duplicadas

In [29]:
num_duplicates = df.duplicated().sum()
print(f"\nNúmero de filas duplicadas: {num_duplicates}")


Número de filas duplicadas: 0


# 4. Valores únicos y posibles inconsistencias en columnas categóricas

In [30]:
print("\nValores Únicos en Columnas Categóricas (y conteo si son pocas categorías):")

for col in df.select_dtypes(include=['object']).columns:
    unique_vals = df[col].nunique()
    if unique_vals < 30: # Mostrar conteo si hay hasta 30 categorías para no saturar
        print(f"- '{col}': {unique_vals} valores únicos")
        print(f"  Distribución:\n{df[col].value_counts().head(10)}") # Mostrar top 10
    else: # Solo mostrar el número de únicos si son muchos
        print(f"- '{col}': {unique_vals} valores únicos (demasiados para listar todos)")


Valores Únicos en Columnas Categóricas (y conteo si son pocas categorías):
- 'fecha_y_hora': 8784 valores únicos (demasiados para listar todos)


# Proceso de Limpieza y Transformación

In [38]:
df_cleaned = df.copy()

# Manejo de Duplicados

In [39]:
initial_rows_clean = df_cleaned.shape[0]
df_cleaned.drop_duplicates(inplace=True)
rows_after_duplicates_clean = df_cleaned.shape[0]
print(f"Filas eliminadas por duplicidad: {initial_rows_clean - rows_after_duplicates_clean}")
print(f"Filas restantes: {rows_after_duplicates_clean}")

Filas eliminadas por duplicidad: 0
Filas restantes: 8784


#  Conversión de Tipos de Datos (Fecha y Hora)

In [40]:
if 'fecha_y_hora' in df_cleaned.columns:
   
    df_cleaned['fecha_y_hora'] = pd.to_datetime(df_cleaned['fecha_y_hora'], errors='coerce', infer_datetime_format=True)
    if df_cleaned['fecha_y_hora'].isnull().any():
        print("¡Advertencia! Se encontraron fechas/horas inválidas en 'fecha_y_hora' y se convirtieron a NaT.")
    print("Columna 'fecha_y_hora' convertida a tipo datetime.")
else:
    print("La columna 'fecha_y_hora' no fue encontrada. Asegúrate del nombre exacto después de limpiar columnas.")

C:\Users\Diego\AppData\Local\Temp\ipykernel_18976\1239691205.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_cleaned['fecha_y_hora'] = pd.to_datetime(df_cleaned['fecha_y_hora'], errors='coerce', infer_datetime_format=True)
C:\Users\Diego\AppData\Local\Temp\ipykernel_18976\1239691205.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_cleaned['fecha_y_hora'] = pd.to_datetime(df_cleaned['fecha_y_hora'], errors='coerce', infer_datetime_format=True)


Columna 'fecha_y_hora' convertida a tipo datetime.


# Manejo de Valores Faltantes

In [41]:
# Eliminar columnas con demasiados valores faltantes
columns_to_drop = ['no2_g_m3', 'no_g_m3', 'nox_g_m3', 'co_g_m3', 'so2_g_m3']

# Verificar qué columnas existen antes de eliminarlas
existing_columns_to_drop = [col for col in columns_to_drop if col in df_cleaned.columns]

if existing_columns_to_drop:
    print(f"Eliminando columnas con demasiados valores faltantes: {existing_columns_to_drop}")
    df_cleaned.drop(columns=existing_columns_to_drop, inplace=True)
    print(f"Columnas eliminadas exitosamente. Dimensiones actuales: {df_cleaned.shape}")
else:
    print("Ninguna de las columnas especificadas fue encontrada en el dataset.")

# Mostrar las columnas restantes
print(f"\nColumnas restantes ({len(df_cleaned.columns)}):")
print(df_cleaned.columns.tolist())

Eliminando columnas con demasiados valores faltantes: ['no2_g_m3', 'no_g_m3', 'nox_g_m3', 'co_g_m3', 'so2_g_m3']
Columnas eliminadas exitosamente. Dimensiones actuales: (8784, 12)

Columnas restantes (12):
['fecha_y_hora', 'presion_atmosferica_mm_hg', 'temperatura_ambiente_celsius', 'velocidad_del_viento_m_s', 'direccion_del_viento', 'humedad_relativa_percent', 'radiacion_solar_w_m2', 'precipitacion_mm', 'o3_g_m3', 'trs_g_m3', 'pm10_g_m3', 'pm2_5_g_m3']


In [42]:
for col in df_cleaned.columns:
    if df_cleaned[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df_cleaned[col]):
            median_val = df_cleaned[col].median()
            df_cleaned[col].fillna(median_val, inplace=True)
            print(f"Valores faltantes en '{col}' (numérica) imputados con la mediana: {median_val:.2f}")
        elif df_cleaned[col].dtype == 'object':
            # Si hay columnas object que no son fecha/hora y tienen nulos
            df_cleaned[col].fillna('DESCONOCIDO', inplace=True)
            print(f"Valores faltantes en '{col}' (categórica) imputados con 'DESCONOCIDO'.")
        elif pd.api.types.is_datetime64_any_dtype(df_cleaned[col]):
            
            initial_rows_dt = df_cleaned.shape[0]
            df_cleaned.dropna(subset=[col], inplace=True)
            print(f"Filas con valores NaT en '{col}' eliminadas: {initial_rows_dt - df_cleaned.shape[0]}")


if df_cleaned.isnull().sum().sum() == 0:
    print("¡Todos los valores faltantes han sido manejados!")
else:
    print("Todavía quedan valores faltantes después de la imputación:")
    print(df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0])

Valores faltantes en 'presion_atmosferica_mm_hg' (numérica) imputados con la mediana: 694.67
Valores faltantes en 'temperatura_ambiente_celsius' (numérica) imputados con la mediana: 24.79
Valores faltantes en 'velocidad_del_viento_m_s' (numérica) imputados con la mediana: 1.49
Valores faltantes en 'direccion_del_viento' (numérica) imputados con la mediana: 224.52
Valores faltantes en 'humedad_relativa_percent' (numérica) imputados con la mediana: 78.71
Valores faltantes en 'radiacion_solar_w_m2' (numérica) imputados con la mediana: 3.12
Valores faltantes en 'precipitacion_mm' (numérica) imputados con la mediana: 0.00
Valores faltantes en 'o3_g_m3' (numérica) imputados con la mediana: 25.58
Valores faltantes en 'trs_g_m3' (numérica) imputados con la mediana: 1.79
Valores faltantes en 'pm10_g_m3' (numérica) imputados con la mediana: 19.30
Valores faltantes en 'pm2_5_g_m3' (numérica) imputados con la mediana: 9.81
¡Todos los valores faltantes han sido manejados!


C:\Users\Diego\AppData\Local\Temp\ipykernel_18976\1353617188.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned[col].fillna(median_val, inplace=True)
C:\Users\Diego\AppData\Local\Temp\ipykernel_18976\1353617188.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

# Creación de Nuevas Características

In [43]:
# Extraer componentes de la fecha y hora
if 'fecha_y_hora' in df_cleaned.columns:
    df_cleaned['año'] = df_cleaned['fecha_y_hora'].dt.year
    df_cleaned['mes'] = df_cleaned['fecha_y_hora'].dt.month
    df_cleaned['dia'] = df_cleaned['fecha_y_hora'].dt.day
    df_cleaned['hora'] = df_cleaned['fecha_y_hora'].dt.hour
    df_cleaned['dia_de_la_semana'] = df_cleaned['fecha_y_hora'].dt.dayofweek # Lunes=0, Domingo=6
    df_cleaned['nombre_dia_semana'] = df_cleaned['fecha_y_hora'].dt.day_name(locale='es') # Nombre del día en español
    print("Columnas de tiempo (año, mes, día, hora, día_de_la_semana, nombre_dia_semana) creadas.")
else:
    print("La columna 'fecha_y_hora' no está disponible.")

Columnas de tiempo (año, mes, día, hora, día_de_la_semana, nombre_dia_semana) creadas.


# Estado Final del Dataset (Limpio y Transformado)

In [44]:
print("Primeras 5 filas del dataset limpio:")
print(df_cleaned.head())

Primeras 5 filas del dataset limpio:
         fecha_y_hora  presion_atmosferica_mm_hg  \
0 2024-01-01 00:00:00                    696.291   
1 2024-01-01 01:00:00                    696.132   
2 2024-01-01 02:00:00                    695.799   
3 2024-01-01 03:00:00                    695.300   
4 2024-01-01 04:00:00                    694.951   

   temperatura_ambiente_celsius  velocidad_del_viento_m_s  \
0                        24.086                     1.142   
1                        23.734                     1.420   
2                        23.169                     1.592   
3                        23.281                     1.368   
4                        22.887                     2.341   

   direccion_del_viento  humedad_relativa_percent  radiacion_solar_w_m2  \
0               181.289                    83.830                   0.0   
1               121.161                    82.638                   0.0   
2                68.523                    86.201         

In [45]:
print("\nInformación del dataset limpio (tipos de datos y valores no nulos):")
df_cleaned.info()


Información del dataset limpio (tipos de datos y valores no nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   fecha_y_hora                  8784 non-null   datetime64[ns]
 1   presion_atmosferica_mm_hg     8784 non-null   float64       
 2   temperatura_ambiente_celsius  8784 non-null   float64       
 3   velocidad_del_viento_m_s      8784 non-null   float64       
 4   direccion_del_viento          8784 non-null   float64       
 5   humedad_relativa_percent      8784 non-null   float64       
 6   radiacion_solar_w_m2          8784 non-null   float64       
 7   precipitacion_mm              8784 non-null   float64       
 8   o3_g_m3                       8784 non-null   float64       
 9   trs_g_m3                      8784 non-null   float64       
 10  pm10_g_m3                  

In [46]:
print("\nEstadísticas descriptivas de columnas numéricas del dataset limpio:")
print(df_cleaned.describe())


Estadísticas descriptivas de columnas numéricas del dataset limpio:
              fecha_y_hora  presion_atmosferica_mm_hg  \
count                 8784                8784.000000   
mean   2024-07-01 23:30:00                 694.594552   
min    2024-01-01 00:00:00                 690.607000   
25%    2024-04-01 11:45:00                 693.728750   
50%    2024-07-01 23:30:00                 694.674000   
75%    2024-10-01 11:15:00                 695.499250   
max    2024-12-31 23:00:00                 698.510000   
std                    NaN                   1.266819   

       temperatura_ambiente_celsius  velocidad_del_viento_m_s  \
count                   8784.000000               8784.000000   
mean                      25.132557                  2.037836   
min                       19.410000                  0.432000   
25%                       22.819500                  1.051000   
50%                       24.793000                  1.493000   
75%                       2

# Métricas clave del estado final:

In [47]:
# 1. Número total de filas y columnas
print(f"\nDimensiones del dataset limpio: {df_cleaned.shape[0]} filas, {df_cleaned.shape[1]} columnas")


Dimensiones del dataset limpio: 8784 filas, 18 columnas


In [51]:
# 2. Conteo de valores faltantes por columna (debería ser 0 para las columnas procesadas)
missing_values_cleaned = df_cleaned.isnull().sum()
missing_percentage_cleaned = (df_cleaned.isnull().sum() / len(df_cleaned)) * 100
missing_df_cleaned = pd.DataFrame({'Missing Count': missing_values_cleaned, 'Missing %': missing_percentage_cleaned})
print("\nValores Faltantes por Columna :")
print(missing_df_cleaned[missing_df_cleaned['Missing Count'] > 0])
if missing_df_cleaned[missing_df_cleaned['Missing Count'] > 0].empty:
    print("¡No hay valores faltantes en el dataset ! ")


Valores Faltantes por Columna :
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []
¡No hay valores faltantes en el dataset ! 


In [ ]:
# 3. Número de filas duplicadas
num_duplicates_cleaned = df_cleaned.duplicated().sum()
print(f"\nNúmero de filas duplicadas : {num_duplicates_cleaned}")
if num_duplicates_cleaned == 0:
    print("¡No hay filas duplicadas en el dataset ! ")


Número de filas duplicadas : 0
¡No hay filas duplicadas en el dataset ! 


In [ ]:
# 4. Valores únicos y consistencia en columnas categóricas (después de estandarización)
print("\nValores Únicos en Columnas Categóricas:")
for col in df_cleaned.select_dtypes(include=['object', 'category']).columns:
    unique_vals_cleaned = df_cleaned[col].nunique()
    if unique_vals_cleaned < 30: # Mostrar conteo si hay hasta 30 categorías
        print(f"- '{col}': {unique_vals_cleaned} valores únicos")
        print(f"  Distribución:\n{df_cleaned[col].value_counts().head(10)}")
    else:
        print(f"- '{col}': {unique_vals_cleaned} valores únicos (demasiados para listar)")


Valores Únicos en Columnas Categóricas (Dataset Limpio):
- 'nombre_dia_semana': 7 valores únicos
  Distribución:
nombre_dia_semana
Lunes        1272
Martes       1272
Miércoles    1248
Jueves       1248
Viernes      1248
Sábado       1248
Domingo      1248
Name: count, dtype: int64
